# IDR regression calibrator usage

This notebook is the canonical usage reference for `calibrated-explanations-calibration-idr`. The plugin is a **post-hoc interval calibrator** over scalar predictions from an underlying regression model. It is not the underlying model.


## Lifecycle rules

Valid pattern A: CE owns fitting of the underlying model via `explainer.fit(...)`.

Valid pattern B: the model is already fitted; in that case, do **not** call `explainer.fit(...)`.

> Do not call `model.fit(...)` and `explainer.fit(...)` on the same model instance. That is a double-fit anti-pattern.


In [ ]:
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from ce_calibration_idr import IDRRegressionIntervalCalibratorPlugin
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


In [ ]:
X, y = make_regression(
    n_samples=240,
    n_features=6,
    noise=8.0,
    random_state=0,
)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.4, random_state=0
)
X_cal, X_test, y_cal, _ = train_test_split(
    X_holdout, y_holdout, test_size=0.5, random_state=0
)
plugin_id = IDRRegressionIntervalCalibratorPlugin.plugin_meta["name"]


## Pattern A: CE owns model fitting


In [ ]:
model = RandomForestRegressor(random_state=0)

explainer = WrapCalibratedExplainer(model)
explainer.fit(X_train, y_train)
explainer.calibrate(
    X_cal,
    y_cal,
    interval_calibrator=plugin_id,
)

regression_explanations = explainer.explain_factual(
    X_test[:5],
    low_high_percentiles=(5, 95),
)
threshold_explanations = explainer.explain_factual(
    X_test[:5],
    threshold=float(np.median(y_cal)),
)


In [ ]:
try:
    regression_explanations.plot(uncertainty=True)
except Exception:  # noqa: BLE001
    pass  # plotting requires matplotlib; skip in headless/CI environments


In [ ]:
try:
    threshold_explanations.plot(uncertainty=True)
except Exception:  # noqa: BLE001
    pass  # plotting requires matplotlib; skip in headless/CI environments


## Pattern B: pre-fitted model

The model is already fitted here, so the wrapper is calibrated directly. Do not call `explainer.fit(...)` in this pattern.


In [ ]:
prefit_model = RandomForestRegressor(random_state=0).fit(X_train, y_train)

prefit_explainer = WrapCalibratedExplainer(prefit_model)
prefit_explainer.calibrate(
    X_cal,
    y_cal,
    interval_calibrator=plugin_id,
)
prefit_explanations = prefit_explainer.explain_factual(
    X_test[:5],
    low_high_percentiles=(5, 95),
)
try:
    prefit_explanations.plot(uncertainty=True)
except Exception:  # noqa: BLE001
    pass  # plotting requires matplotlib; skip in headless/CI environments


In [ ]:
no_plugin_explainer = WrapCalibratedExplainer(prefit_model)
no_plugin_explainer.calibrate(
    X_cal,
    y_cal,
)
no_plugin_explanations = no_plugin_explainer.explain_factual(
    X_test[:5],
    low_high_percentiles=(5, 95),
)
try:
    no_plugin_explanations.plot(uncertainty=True)
except Exception:  # noqa: BLE001
    pass  # plotting requires matplotlib; skip in headless/CI environments


## Invalid double-fit anti-pattern

Do not run this pattern:


In [ ]:
# Invalid: do not do this.
# model = RandomForestRegressor(random_state=0).fit(X_train, y_train)
# explainer = WrapCalibratedExplainer(model)
# explainer.fit(X_train, y_train)  # double-fit anti-pattern


## Semantics recap

- Ordinary regression: `predict` is the calibrated distribution median, and `low`/`high` are y-space quantiles.
- Thresholded regression: IDR CDF values become event scores, but final probability intervals come from CE Venn-Abers.
- `raw_predict` is diagnostic metadata only.
- The backend is upstream `isodistrreg.IDR`; install compatible Python bindings before running the plugin.
